# Этап 2: Инженерия признаков

Создаем 30+ признаков для детекции фейковых отзывов

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from features import FeatureEngineering

sns.set_style('whitegrid')
%matplotlib inline

## 2.1 Загрузка размеченных данных

In [ ]:
df = pd.read_json('../data/processed/reviews_labeled.json')

print(f"Загружено {len(df)} отзывов")
print(f"\nРаспределение классов:")
print(df['label'].value_counts())
print(f"\nБаланс: {df['label'].value_counts(normalize=True)}")

df.head()

## 2.2 Создание признаков

In [ ]:
fe = FeatureEngineering()
df_features = fe.create_all_features(df)

print(f"\nСоздано признаков: {len(df_features.columns)}")
print(f"Форма датасета: {df_features.shape}")

In [ ]:
# Список всех признаков
feature_cols = [col for col in df_features.columns 
                if col not in ['text', 'nmId', 'date', 'label']]

print(f"Всего признаков: {len(feature_cols)}")
print("\nСписок признаков:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i}. {col}")

## 2.3 Анализ признаков

In [ ]:
# Статистика по ключевым признакам
key_features = ['text_length', 'word_count', 'fake_keywords_count', 
                'exclamation_count', 'unique_words_ratio', 'rating']

df_features.groupby('label')[key_features].mean()

In [ ]:
# Визуализация распределений
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    ax = axes[idx]
    
    df_features[df_features['label'] == 0][feature].hist(
        ax=ax, alpha=0.5, label='Real', bins=30, color='green'
    )
    df_features[df_features['label'] == 1][feature].hist(
        ax=ax, alpha=0.5, label='Fake', bins=30, color='red'
    )
    
    ax.set_title(feature)
    ax.legend()
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Корреляция признаков с целевой переменной
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
correlations = df_features[numeric_cols].corr()['label'].sort_values(ascending=False)

print("Топ-15 признаков по корреляции с label:")
print(correlations[1:16])  # Исключаем саму label

# Визуализация
plt.figure(figsize=(10, 8))
correlations[1:16].plot(kind='barh', color='steelblue')
plt.title('Топ-15 признаков по корреляции с целевой переменной')
plt.xlabel('Correlation with label')
plt.tight_layout()
plt.savefig('../reports/figures/feature_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.4 Сохранение данных

In [ ]:
# Сохраняем признаковую матрицу
df_features.to_csv('../data/features/features.csv', index=False)

print(f"✓ Признаки сохранены в data/features/features.csv")
print(f"Размер: {df_features.shape}")

## 2.5 Выводы

**Ключевые наблюдения:**

1. Фейковые отзывы в среднем короче реальных
2. Фейки содержат больше ключевых слов-маркеров
3. Реальные отзывы более разнообразны по лексике
4. Фейки чаще имеют максимальный рейтинг (5)

Эти признаки будут использованы для обучения моделей.